In [1]:
import os

os.environ["HF_HOME"] = "~/luiz.pereira/.huggingface"

In [61]:
import datasets
import torch
from model_depth import ParsingNet
from transformers import AutoModel, AutoTokenizer


class DMRSTParser:

  def __init__(self,
               model_path: str = 'checkpoint/multi_all_checkpoint.torchsave'):

    bert_model = AutoModel.from_pretrained("xlm-roberta-base")

    bert_model = bert_model.cuda()

    for _, param in bert_model.named_parameters():
      param.requires_grad = False

    self.model_path = model_path

    self.tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base",
                                                   use_fast=True)

    self.model = ParsingNet(bert_model, bert_tokenizer=self.tokenizer)

    self.model = self.model.cuda()
    self.state_dict = torch.load(self.model_path)
    self.position_ids = self.state_dict.pop(
        'encoder.language_model.embeddings.position_ids')
    self.model.load_state_dict(self.state_dict)
    self.model = self.model.eval()

  def inference(self, examples, field="story") -> None:
    texts = examples[field]

    input_sentences = [
        self.tokenizer.tokenize(i, add_special_tokens=False) for i in texts
    ]
    with torch.no_grad():
      _, _, all_tree_parsing_pred, _, predict_EDU_breaks = self.model.TestingLoss(
          input_sentences,
          input_EDU_breaks=None,
          LabelIndex=None,
          ParsingIndex=None,
          GenerateTree=True,
          use_pred_segmentation=True)

    all_edus = []
    for idx, edu_x in enumerate(predict_EDU_breaks):
      current_edu = 0
      edus = ["" for _ in range(len(edu_x))]
      for i, element in enumerate(input_sentences[idx]):
        if i <= predict_EDU_breaks[idx][current_edu]:
          if i == 0:
            edus[current_edu] += element.replace("▁", "")
          elif element.find("▁") == -1:
            edus[current_edu] += element
          else:
            edus[current_edu] += " " + element.replace("▁", "")
        else:
          current_edu += 1
          edus[current_edu] += element.replace("▁", "")
      all_edus.append(edus)
    rst = sum(all_tree_parsing_pred, [])
    examples["edus"] = all_edus
    examples["edus_breaks"] = predict_EDU_breaks
    examples["rst"] = [r.strip().split() for r in rst]
    return examples

In [62]:
parser = DMRSTParser(
    model_path=
    "/home/luiz.pereira/cohereclassifier/dmrst_parser/checkpoint/multi_all_checkpoint.torchsave"
)

In [48]:
dataset = datasets.load_from_disk('/hadatasets/luiz.pereira/commom_stories_v2')
dataset

Dataset({
    features: ['story_id', 'story', 'plot', 'generated_from', 'generated', 'dataset'],
    num_rows: 487272
})

In [63]:
parser.inference(dataset[0:2])

{'story_id': ['5c4b0f64-2b82-4cbe-8808-5962d0d32d1d',
  '14b6b59f-d648-4fbd-b6e8-8dca4238f8a1'],
 'story': ['I started my first campfire the other night. First I gathered wood and kindling. Next I arranged the wood and kindling in the fire pit. I then lit the fire. Then we had an amazing campfire!',
  'Frederick had never been on a cruise ship before. His family was on vacation. They were cruising in the Caribbean. Frederick had a great time on board going swimming and playing games. He would always remember that first cruise.'],
 'plot': ['|| started first campfire || gathered wood kindling || arranged wood kindling fire pit || lit fire || had amazing campfire ',
  '|| Frederick never cruise ship before || family on vacation || cruising Caribbean || Frederick had great time on board || going swimming and playing games || he would always remember that first cruise '],
 'generated_from': ['5c4b0f64-2b82-4cbe-8808-5962d0d32d1d',
  '14b6b59f-d648-4fbd-b6e8-8dca4238f8a1'],
 'generated': [F